In [2]:
import pyarrow.parquet as pq
import pyarrow as pa

In [13]:
table = pq.read_table("bitbrains128/tasks.parquet.bak")

# 2️⃣ Modify a specific column (e.g., convert to uppercase if it's a string column)
new_column = table["start_time"].to_pandas()

new_schema = table.schema.set(
    table.schema.get_field_index("start_time"),
    pa.field("submission_time", type=table.schema.field("start_time").type)
)

new_schema = pa.schema([
    pa.field(name, field.type, nullable=False if name == "submission_time" else field.nullable)
    for name, field in zip(new_schema.names, new_schema)
])

table = table.set_column(
    table.schema.get_field_index("start_time"),  # Get column index
    "submission_time",  # Column name
    pa.array(new_column, type=table.schema.field("start_time").type,)# Convert back to PyArrow Array
)
updated_table = pa.Table.from_arrays(table.columns, schema=new_schema)

# # 3️⃣ Replace the column in the Table

pq.write_table(updated_table, "bitbrains128/tasks.parquet")

updated_table.schema



cpu_capacity: double not null
cpu_count: int32 not null
id: string not null
mem_capacity: int64 not null
submission_time: timestamp[ms] not null
stop_time: timestamp[ms] not null